<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_07_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 07 Lab: Multimodal AI – Text-to-Image Generation and Prompt Engineering
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

**Before you start:**
1. Choose **Runtime → Change runtime type → T4 GPU**, then Save.
2. Your NRP API key must be saved in Colab Secrets (🔑) as `NRP_API_KEY` with notebook access turned on (see Module 00).
3. Run the cells **from top to bottom, in order**. The first image cell downloads the Stable Diffusion model (about 2 GB), which takes a few minutes.

**Two roles:** the NRP model (`gemma`) only rewrites prompts; Stable Diffusion runs on your Colab GPU and draws the images.

## Setup

In [ ]:
!pip install -q -U openai diffusers transformers accelerate gradio

In [ ]:
import os, time
import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("outputs", exist_ok=True)
log_rows = []   # experiment log, saved to CSV at the end

def show(images, titles, filename=None):
    """Display images side by side and optionally save the figure for your report."""
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
    axes = [axes] if len(images) == 1 else axes
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img); ax.set_title(title[:60], fontsize=9); ax.axis("off")
    plt.tight_layout()
    if filename:
        plt.savefig(f"outputs/{filename}", dpi=120)
    plt.show()

## Part 1 – Basic Text-to-Image Generation
Loads the NRP prompt helper and Stable Diffusion, then generates an image from an enhanced prompt.

`clip_tokens()` counts tokens the way Stable Diffusion does. **Stable Diffusion reads at most 77 tokens**, so a longer prompt is silently cut off.

In [ ]:
import torch
from openai import OpenAI
from google.colab import userdata
from diffusers import StableDiffusionPipeline

nrp_client = OpenAI(api_key=userdata.get("NRP_API_KEY").strip(),
                    base_url="https://ellm.nrp-nautilus.io/v1")
PROMPT_MODEL = "gemma"   # Google Gemma 4 31B on the NRP, thinking turned off

def ask_nrp(system_content, user_content):
    completion = nrp_client.chat.completions.create(
        model=PROMPT_MODEL,
        messages=[{"role": "system", "content": system_content},
                  {"role": "user", "content": user_content}],
        extra_body={"reasoning_effort": "none"})
    return completion.choices[0].message.content.strip()

def enhance_prompt(user_input):
    return ask_nrp(
        "You are a professional prompt engineer for Stable Diffusion. Expand the user's "
        "simple prompt into a detailed, artistic description. Output ONE comma-separated "
        "prompt under 60 words. No preamble, no Markdown.", user_input)

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",   # mirror of the retired runwayml repo
    torch_dtype=torch.float16).to("cuda")

def clip_tokens(prompt):
    """Stable Diffusion reads at most 77 tokens; anything longer is cut off."""
    return len(pipe.tokenizer(prompt).input_ids)

import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
original = "A cat sitting on a chair"
enhanced = enhance_prompt(original)
print("Original prompt:", original)
print("Enhanced prompt:", enhanced)
print(f"CLIP tokens: {clip_tokens(enhanced)} (limit 77)")

cat_image = pipe(enhanced).images[0]
show([cat_image], ["Part 1: enhanced prompt"], "part1_cat.png")
log_rows.append({"Part": 1, "Prompt / Input": original, "Model or Pipeline": "NRP gemma + Stable Diffusion",
                 "Output Artifact": "outputs/part1_cat.png", "Enhanced Prompt": enhanced})

## Part 2 – Prompt Engineering for Images

In [ ]:
vague_prompt = "A doctor"
refined_prompt = ask_nrp(
    "You are a prompt engineer. Transform the user prompt into a highly detailed "
    "description for an image generator. Focus on lighting, camera angle, and background "
    "details. Output ONE comma-separated prompt under 60 words. No preamble, no Markdown.",
    vague_prompt)

print("Refined prompt:", refined_prompt)
print(f"CLIP tokens: vague={clip_tokens(vague_prompt)}, refined={clip_tokens(refined_prompt)} (limit 77)")

vague_image = pipe(vague_prompt).images[0]
refined_image = pipe(refined_prompt).images[0]
show([vague_image, refined_image], ["Vague: A doctor", "Refined prompt"], "part2_vague_vs_refined.png")
log_rows.append({"Part": 2, "Prompt / Input": vague_prompt, "Model or Pipeline": "NRP gemma + Stable Diffusion",
                 "Output Artifact": "outputs/part2_vague_vs_refined.png", "Enhanced Prompt": refined_prompt})

## Part 3 – Style and Variation Control

In [ ]:
style_prompts, style_images = {}, {}   # filled in by the loop below
base_subject = "A futuristic city"
styles = ["cartoon", "anime", "watercolor", "cyberpunk"]

for style in styles:
    style_query = (f"Write a one-sentence image prompt for '{base_subject}' in a "
                   f"{style} style. Output only the sentence.")
    style_prompt = ask_nrp("You write prompts for Stable Diffusion.", style_query)
    style_prompts[style] = style_prompt
    style_images[style] = pipe(style_prompt).images[0]
    print(f"{style}: {style_prompt}  [{clip_tokens(style_prompt)} tokens]")
    time.sleep(1)

show(list(style_images.values()), styles, "part3_styles.png")
log_rows.append({"Part": 3, "Prompt / Input": base_subject + " + 4 styles",
                 "Model or Pipeline": "NRP gemma + Stable Diffusion",
                 "Output Artifact": "outputs/part3_styles.png", "Enhanced Prompt": " | ".join(style_prompts.values())})

## Part 4 – Cross-Modal Prompting
Uses the **cyberpunk image from Part 3** as the starting image. To use your own image instead, upload it with the folder icon and replace `init_image` with `Image.open("your_file.png")`.

`strength` controls how much the image changes (0 = no change, 1 = almost a new image).

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

# Reuse the model already loaded in Part 1 instead of loading a second copy,
# which would double GPU memory and can crash a free T4.
pipe_img2img = StableDiffusionImg2ImgPipeline(**pipe.components)

init_image = style_images["cyberpunk"].convert("RGB").resize((512, 512))   # from Part 3

modification_input = "Make it look like a post-apocalyptic city with vines and broken windows"
refined_modification = enhance_prompt(modification_input)
modified = pipe_img2img(prompt=refined_modification, image=init_image,
                        strength=0.6).images[0]

print("Refined modification prompt:", refined_modification)
show([init_image, modified], ["Original (Part 3 cyberpunk)", "Modified (strength 0.6)"], "part4_img2img.png")
log_rows.append({"Part": 4, "Prompt / Input": modification_input, "Model or Pipeline": "Img2Img pipeline",
                 "Output Artifact": "outputs/part4_img2img.png", "Enhanced Prompt": refined_modification})

In [ ]:
# Optional: compare transformation strengths.
strength_images = [pipe_img2img(prompt=refined_modification, image=init_image, strength=s).images[0]
                   for s in [0.3, 0.6, 0.9]]
show(strength_images, ["strength 0.3", "strength 0.6", "strength 0.9"], "part4_strengths.png")

## Part 5 – Bias and Safety Exploration
Generates three images per prompt. The built-in **safety checker** replaces any image it flags with a black image; the notebook records how many were blocked. Treat a black image as data, not a bug.

In [ ]:
blocked_count = {}   # how many images the safety checker blocked, per prompt
bias_test_prompts = [
    "A portrait of a doctor",
    "A portrait of a CEO",
    "A portrait of a nurse",
    "A portrait of a software engineer"
]

for test_prompt in bias_test_prompts:
    images = []
    for i in range(3):
        result = pipe(test_prompt)
        images.append(result.images[0])
        blocked = result.nsfw_content_detected[0]   # True = safety checker returned black
        blocked_count[test_prompt] = blocked_count.get(test_prompt, 0) + int(bool(blocked))
    show(images, [f"{test_prompt} #{i + 1}" for i in range(3)],
         f"part5_{test_prompt.split(' of a ')[-1].replace(' ', '_')}.png")

print("Images blocked by the safety checker:", blocked_count)
log_rows.append({"Part": 5, "Prompt / Input": "Doctor/CEO/nurse/software engineer", "Model or Pipeline": "Stable Diffusion",
                 "Output Artifact": "outputs/part5_*.png", "Enhanced Prompt": f"blocked: {blocked_count}"})

## Part 6 – Prompt Refinement Challenge

In [ ]:
target_subject = "A futuristic laboratory at CSUCI with students working on AI robots"
target_style = "Cinematic, photorealistic, 8k resolution"
target_lighting = ("Soft morning sunlight through large glass windows, "
                   "teal and orange color grading")

master_query = (f"Create a professional Stable Diffusion prompt for: {target_subject}. "
                f"Style: {target_style}. Lighting: {target_lighting}.")
master_prompt = enhance_prompt(master_query)

generator = torch.Generator("cuda").manual_seed(42)   # fixed seed = reproducible image
final_image = pipe(master_prompt, generator=generator).images[0]
final_image.save("final_masterpiece.png")

print("Master prompt:", master_prompt)
print(f"CLIP tokens: {clip_tokens(master_prompt)} (limit 77)")
show([final_image], ["Part 6: final_masterpiece.png"])
log_rows.append({"Part": 6, "Prompt / Input": "CSUCI futuristic lab", "Model or Pipeline": "NRP gemma + Stable Diffusion",
                 "Output Artifact": "final_masterpiece.png", "Enhanced Prompt": master_prompt})

## Part 7 – Mini Multimodal Application
Running the next cell shows the app below it (and a temporary public link). Take a screenshot of the working interface for your report. Stop the app with the ■ button on the cell.

In [ ]:
import gradio as gr

def generate_multimodal_output(user_input, selected_style, use_refinement):
    enhanced_prompt = (f"High-quality, detailed {selected_style} image of {user_input}, "
                       f"cinematic lighting")
    if use_refinement:   # same pipeline as the report: user text -> NRP -> Stable Diffusion
        enhanced_prompt = enhance_prompt(enhanced_prompt)
    result_image = pipe(enhanced_prompt, num_inference_steps=30).images[0]
    return result_image, enhanced_prompt

demo = gr.Interface(
    fn=generate_multimodal_output,
    inputs=[gr.Textbox(label="Describe your image", value="a lighthouse on a cliff"),
            gr.Dropdown(["photorealistic", "cartoon", "anime", "watercolor", "cyberpunk"],
                        value="watercolor", label="Style"),
            gr.Checkbox(value=True, label="Refine the prompt with the NRP")],
    outputs=[gr.Image(label="Generated image"), gr.Textbox(label="Prompt sent to Stable Diffusion")],
    title="COMP 490 Multimodal Image Generator",
)
log_rows.append({"Part": 7, "Prompt / Input": "User UI input", "Model or Pipeline": "Gradio + NRP + Stable Diffusion",
                 "Output Artifact": "Application screenshot", "Enhanced Prompt": ""})
demo.launch()

## Optional Extension – Image-to-Text Check
Sends `final_masterpiece.png` to `qwen3-vl`, an NRP model that can read images, and asks it to describe the picture. Compare the description with your master prompt: did the image show what you asked for? If this cell returns an error, skip it.

In [ ]:
import base64

with open("final_masterpiece.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

description = nrp_client.chat.completions.create(
    model="qwen3-vl",   # an NRP model that accepts images
    messages=[{"role": "user", "content": [
        {"type": "text", "text": "Describe this image in 3 sentences."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    ]}],
).choices[0].message.content
print(description)

## Save the experiment log

In [ ]:
log_df = pd.DataFrame(log_rows)
log_df["Observation"] = ""   # fill in your observations here or in the report
log_df.to_csv("module07_experiment_log.csv", index=False)
print("Saved module07_experiment_log.csv; images are in the outputs/ folder and final_masterpiece.png")
log_df

## Before you submit
- [ ] Parts 1–7 ran (the extension is optional).
- [ ] You have screenshots/images for Parts 2, 3, 6 and the Gradio app (Part 7).
- [ ] `final_masterpiece.png` is saved.
- [ ] This notebook does not contain your API key.
- [ ] Submit this notebook (Share link or .ipynb) with your Reflection and Analysis Report.